# 03 · Stage 2 — model training & Optuna tuning
Default task: Priority vs Best Performer among high-access districts, with a class-1 threshold tuned on out-of-fold predictions for recall ≥ 0.82.
The full tuning run (`python -m src.run_pipeline`) writes `models/benchmark_metrics.json`. This notebook reports that benchmark; set `RETRAIN = True` to re-run training here (50 trials × 4 models takes several minutes).

In [1]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [2]:
RETRAIN = False
if RETRAIN:
    from src.run_pipeline import run
    run(trials=config["modelling"]["optuna_trials"])
metrics = json.loads(P("metrics").read_text())
rows = []
for name, r in metrics["candidate_models"].items():
    h, o = r["holdout"], r["out_of_fold"]
    rows.append({"model": name, "treeshap": r["treeshap_compatible"], "trials": r["optuna_trials"], "cv_objective": r["cv_selection_score"],
                 "oof_c1_recall": o["class_1_recall"], "holdout_c1_recall": h["class_1_recall"], "holdout_c1_f1": h["class_1_f1"],
                 "holdout_c1_f2": h["class_1_f2"], "holdout_macro_f1": h["macro_f1"], "holdout_roc_auc": h["ovr_roc_auc"],
                 "holdout_pr_auc": h["ovr_pr_auc"], "holdout_c1_brier": h.get("class_1_brier"),
                 "holdout_c1_recall@thr": r.get("holdout_at_threshold", {}).get("class_1_recall"),
                 "holdout_c1_precision@thr": r.get("holdout_at_threshold", {}).get("class_1_precision")})
print("Task:", metrics.get("task"), "| decision threshold:", metrics.get("decision_threshold"), "|", metrics.get("threshold_selection", {}).get("rule"))
print("Selected:", metrics["selected_model"], "|", metrics["selection_rule"])
pd.DataFrame(rows).round(3)

Task: discordance_binary | decision threshold: 0.3482 | highest threshold with OOF class-1 recall >= 0.82
Selected: lightgbm | highest mean 5-fold CV objective (0.60 x class-1 F1 + 0.40 x macro F1) among TreeSHAP-compatible models; holdout is reported, never used for selection


,model,treeshap,trials,cv_objective,oof_c1_recall,holdout_c1_recall,holdout_c1_f1,holdout_c1_f2,holdout_macro_f1,holdout_roc_auc,holdout_pr_auc,holdout_c1_brier,holdout_c1_recall@thr,holdout_c1_precision@thr
0,random_forest,True,50,0.575,0.644,0.682,0.566,0.630,0.648,0.734,0.596,0.213,NaN,NaN
1,gradient_boosting,True,50,0.541,0.540,0.636,0.571,0.609,0.668,0.759,0.645,0.193,NaN,NaN
2,xgboost,True,50,0.580,0.678,0.591,0.531,0.565,0.636,0.687,0.602,0.216,NaN,NaN
3,lightgbm,True,50,0.584,0.655,0.636,0.609,0.625,0.707,0.739,0.593,0.197,0.773,0.472


In [3]:
selected = metrics["candidate_models"][metrics["selected_model"]]
cm = selected.get("holdout_at_threshold", selected["holdout"])["confusion_matrix"]
pd.DataFrame(cm["matrix"], index=[f"obs {l}" for l in cm["labels"]], columns=[f"pred {l}" for l in cm["labels"]])

,pred 1,pred 3
obs 1,17,5
obs 3,19,28


In [4]:
cal = metrics["calibration"]
print("Class-1 Brier raw: %.4f -> calibrated: %.4f" % (cal["holdout_class_1_brier_raw"], cal["holdout_class_1_brier_calibrated"]))
selected["best_params"]

Class-1 Brier raw: 0.1970 -> calibrated: 0.1833


{'n_estimators': 300,
 'learning_rate': 0.01891123289157431,
 'num_leaves': 60,
 'min_child_samples': 18,
 'subsample': 0.6249446773966186,
 'colsample_bytree': 0.896167185369753}

## Robustness: state-grouped CV and label sensitivity
Written by `python -m src.models.validate` (also run by the pipeline). Whole states are held out; thresholds are re-chosen inside each training fold.

In [5]:
validation = json.loads(P("validation_report").read_text())
pd.DataFrame({s: r["fold_mean"] for s, r in validation["cv"].items()}).T.round(3)

,class_1_recall,class_1_precision,class_1_f2,class_1_roc_auc
random_stratified,0.835,0.443,0.707,0.742
state_grouped,0.652,0.337,0.535,0.537


In [6]:
pd.DataFrame(validation["per_state_recall"]).round(2)

,state,n_priority,detected,recall_when_state_held_out
0,Punjab,6,1,0.17
1,Tamil Nadu,26,7,0.27
2,West Bengal,4,2,0.50
3,Haryana,13,8,0.62
4,Karnataka,4,3,0.75
5,NCT of Delhi,4,3,0.75
6,Madhya Pradesh,14,11,0.79
7,Odisha,5,4,0.80
8,Maharashtra,4,4,1.00
9,Rajasthan,9,9,1.00


In [7]:
pd.DataFrame(validation["label_sensitivity"]).drop(columns="settings").round(3)

,variant,n_priority,priority_jaccard_vs_baseline,baseline_priority_retained,all_class_agreement
0,baseline,109,1.000,1.000,1.000
1,mortality_only,88,0.628,0.697,0.891
2,care_threshold_80pct,195,0.559,1.000,0.673
3,residual_threshold_0,136,0.801,1.000,0.961
4,residual_threshold_0.5,71,0.651,0.651,0.945
5,eb_prior_1000,105,0.945,0.954,0.972
6,eb_prior_10000,104,0.885,0.917,0.968
